# 10 — GPU Acceleration Demo

AI-Lake detects NVIDIA CUDA and AMD ROCm **at runtime** via `libloading` — no recompile needed.

This notebook covers:
1. `hardware_info()` — inspect detected backend
2. `write_batch_auto` — auto-selects IVF-PQ (GPU) vs HNSW (CPU)
3. Write timing: GPU IVF-PQ vs CPU HNSW at scale
4. Search throughput: GPU flat scan vs CPU flat scan
5. Recall comparison
6. Graceful fallback when no GPU present

> **Note:** All demos degrade gracefully to CPU when running without a GPU.
> On the `--profile gpu` Docker service an NVIDIA GPU is reserved via the NVIDIA Container Toolkit.

> **Verification limitation:** actual CUDA/ROCm kernel execution (cuBLAS/hipBLAS SGEMM,
> GPU k-means for IVF-PQ training) cannot be exercised or confirmed in an environment
> without NVIDIA/AMD GPU hardware — this notebook, run without a GPU, only verifies the
> hardware-detection heuristic and the CPU fallback path end-to-end. The `--profile gpu`
> compose service targets real GPU hardware; run it on a CUDA-capable host to exercise
> the GPU code paths themselves.


## Setup

In [ ]:
import os, json, time, pathlib, tempfile
import numpy as np
import ailake

QUERY_PATH = os.environ.get("QUERY_PATH", "/data/demo_query.json")
with open(QUERY_PATH) as f:
    demo = json.load(f)

DIM   = int(demo["dim"])
Q_VEC = demo["query_vector"]

GPU_DEMO = os.environ.get("AILAKE_GPU_DEMO", "0") == "1"
print(f"GPU_DEMO env: {GPU_DEMO}")
print(f"Vector dim: {DIM}")


## 1. Hardware Detection

`ailake.hardware_info()` returns a dict with the detected backend and CPU topology.

Backend priority order (determined at compile-time detection logic, runtime dlopen):
1. **AMD ROCm** — if `libamdhip64.so` + `libhipblas.so` found
2. **NVIDIA CUDA** — if `libcudart.so` + `libcublas.so` found
3. **CPU SIMD** — always available fallback (AVX2/AVX-512/F16C/FMA)


In [ ]:
info = ailake.hardware_info()

print("=" * 50)
print("AI-Lake Hardware Profile")
print("=" * 50)
print(f"  backend:            {info['backend']}")
print(f"  has_cuda:           {info['has_cuda']}")
print(f"  has_rocm:           {info['has_rocm']}")
print(f"  cpu_logical_cores:  {info['cpu_logical_cores']}")
print(f"  has_avx2:           {info['has_avx2']}")
print(f"  has_avx512:         {info['has_avx512']}")
print(f"  recommend_ivf_pq:   {info['recommend_ivf_pq']}  (for N=5000 vectors)")
print("=" * 50)

backend = info["backend"]
has_gpu = info["has_cuda"] == "true" or info["has_rocm"] == "true"
print(f"\nGPU available: {has_gpu}")
if not has_gpu:
    print("Running on CPU SIMD. write_batch_auto will use HNSW.")
    print("Start with --profile gpu to add NVIDIA GPU to this service.")


## 2. `write_batch_auto` — auto index selection

`write_batch_auto` calls `HardwareProfile::detect().recommend_ivf_pq(n_vectors)`:

| Condition | Index chosen |
|-----------|--------------|
| GPU present (CUDA or ROCm) | IVF-PQ (GPU k-means) |
| CPU ≥ 8 logical cores AND n ≥ 5 000 | IVF-PQ (CPU k-means) |
| Otherwise | HNSW (always available) |

Let's write 10 000 random vectors and observe which index is built.


In [ ]:
N = 10_000
rng = np.random.default_rng(seed=42)
embs  = rng.standard_normal((N, DIM)).astype(np.float32)
texts = [f"doc_{i}" for i in range(N)]

AUTO_PATH = "/data/gpu_auto_10k"

os.makedirs(AUTO_PATH, exist_ok=True)
w = ailake.TableWriter(AUTO_PATH, dim=DIM, metric="cosine")

t0 = time.perf_counter()
# write_batch_auto_deferred: Parquet immediate, index built in background.
w.write_batch_auto_deferred(texts, embs.tolist())
snap_id = w.commit()
elapsed = time.perf_counter() - t0

cores    = int(info["cpu_logical_cores"])
will_ivf = info["recommend_ivf_pq"] == "true"

print(f"Wrote {N} vectors in {elapsed:.2f}s  ({N/elapsed:,.0f} vec/s)")
print(f"snapshot_id = {snap_id}")
print(f"Expected index: {'IVF-PQ' if will_ivf else 'HNSW'}  (GPU={has_gpu}, cores={cores})")
print()
print("Note: index training runs in background after commit returns.")
print("Search will wait for IndexStatus::Ready before executing.")


## 3. Explicit timing comparison: IVF-PQ vs HNSW

Force each engine explicitly — `write_batch` (immediate HNSW) vs
`write_batch_ivf_pq_deferred` (always IVF-PQ, independent of the
`write_batch_auto_deferred` hardware heuristic used in §2) — so the comparison
is deterministic regardless of this machine's core count or GPU presence.

`write_batch_ivf_pq` (no `_deferred` suffix) is the synchronous sibling — blocks until the IVF-PQ index is fully built instead of training it in the background; same forced behavior, used when a caller needs the index ready before `commit()` returns.


In [ ]:
def time_write(path, texts, embs, mode="hnsw", label=""):
    os.makedirs(path, exist_ok=True)
    w = ailake.TableWriter(path, dim=DIM, metric="cosine")
    t0 = time.perf_counter()
    if mode == "hnsw":
        # Immediate HNSW: index built inline during write
        w.write_batch(texts, embs.tolist())
    elif mode == "ivfpq_deferred":
        # write_batch_ivf_pq_deferred: always IVF-PQ, regardless of the
        # write_batch_auto_deferred hardware heuristic (GPU, or CPU cores>=8
        # and N>=5000). Parquet lands immediately (~200k vec/s); index trains
        # in the background. Use this when you specifically want IVF-PQ's
        # smaller footprint (good for S3 sequential-scan workloads) rather
        # than whatever the heuristic would have picked on this machine.
        w.write_batch_ivf_pq_deferred(texts, embs.tolist())
    else:
        raise ValueError(mode)
    snap = w.commit()
    dt = time.perf_counter() - t0
    rate = len(texts) / dt
    print(f"{label:25s}  {dt:6.2f}s  {rate:8,.0f} vec/s  snap={snap}")
    return dt

N = 6_000
rng = np.random.default_rng(0)
embs6k  = rng.standard_normal((N, DIM)).astype(np.float32)
texts6k = [f"doc_{i}" for i in range(N)]

print(f"N={N} vectors, DIM={DIM}")
print("-" * 60)
t_hnsw     = time_write("/data/gpu_bench_hnsw",     texts6k, embs6k, mode="hnsw",           label="HNSW (immediate)")
t_deferred = time_write("/data/gpu_bench_deferred", texts6k, embs6k, mode="ivfpq_deferred",  label="IVF-PQ (forced, async)")

ratio = t_hnsw / t_deferred if t_deferred > 0 else float("inf")
winner = "IVF-PQ (forced, async)" if t_deferred < t_hnsw else "HNSW"
print(f"\n{winner} faster Parquet commit by {ratio:.1f}x at N={N}")
print("(IVF-PQ-deferred wins large N; HNSW wins tiny N where index build < 1s)")


## 4. Search throughput: GPU flat scan vs CPU

For top-k search with ANN index, GPU accelerates the **flat re-ranking** step after IVF probing.

We'll run repeated searches and compare QPS (queries per second).

> `ailake.search()` automatically uses GPU distance kernels when a GPU is available — no flag required.


In [ ]:
BENCH_PATH = "/data/gpu_bench_deferred"  # written in cell-8 via write_batch_ivf_pq_deferred (forced IVF-PQ)

# Warm up
for _ in range(3):
    _ = ailake.search(BENCH_PATH, Q_VEC, top_k=10).to_list()

REPS = 50
rng = np.random.default_rng(7)

t0 = time.perf_counter()
for _ in range(REPS):
    q = rng.standard_normal(DIM).tolist()
    ailake.search(BENCH_PATH, q, top_k=10)
elapsed = time.perf_counter() - t0

qps = REPS / elapsed
print(f"{REPS} searches in {elapsed:.3f}s")
print(f"Throughput: {qps:.1f} QPS  (backend={backend})")
if has_gpu:
    print("GPU distance kernels active (SGEMM via cuBLAS / hipBLAS).")
else:
    print("CPU path active. GPU would accelerate distance computation further.")

## 5. Recall comparison: HNSW vs IVF-PQ

Compute recall@10 for both indexes against exact brute-force results.


In [ ]:
import math

def cosine_dist(a, b):
    a, b = np.array(a), np.array(b)
    return 1.0 - float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

HNSW_PATH     = "/data/gpu_bench_hnsw"
DEFERRED_PATH = "/data/gpu_bench_deferred"

Q_VECS = [np.random.default_rng(i).standard_normal(DIM).tolist() for i in range(20)]
K = 10

def collect_ids(path, q, k):
    return {r["row_id"] for r in ailake.search(path, q, top_k=k).to_list()}

hnsw_hits = 0
def_hits  = 0
overlap   = 0
total     = 0

for q in Q_VECS:
    hnsw_ids = collect_ids(HNSW_PATH, q, K)
    def_ids  = collect_ids(DEFERRED_PATH, q, K)
    # Use HNSW as approximate ground truth for recall of the IVF-PQ table
    hnsw_hits += K
    def_hits  += len(def_ids & hnsw_ids)
    overlap   += len(def_ids & hnsw_ids)
    total     += K

recall_def_vs_hnsw = overlap / total if total > 0 else 0.0

print(f"Recall@{K} of IVF-PQ (forced, §3) vs HNSW (approximate ground truth):")
print(f"  IVF-PQ vs HNSW: {recall_def_vs_hnsw:.3f}")
print()
print("Note: both are ANN — HNSW is used as proxy ground truth, not exact neighbors.")
print("Expect ~0.90-0.99 for well-trained IVF-PQ on 6k vectors.")


## 6. Graceful fallback without GPU

AI-Lake never hard-errors on missing GPU. When GPU libs are absent:
- `hardware_info()["has_cuda"]` → `"false"`, `backend` → `"cpu-simd"`
- `write_batch_auto` → selects HNSW (or CPU IVF-PQ if ≥ 8 cores and N ≥ 5000)
- `search()` → CPU SIMD distance kernels (AVX2/AVX-512 when available)

This cell shows the exact fallback logic to verify the runtime detection path.


In [ ]:
info = ailake.hardware_info()

backend = info["backend"]
has_gpu = info["has_cuda"] == "true" or info["has_rocm"] == "true"
cores   = int(info["cpu_logical_cores"])
avx2    = info["has_avx2"] == "true"
avx512  = info["has_avx512"] == "true"

print("Runtime detection summary")
print("-" * 40)
print(f"Backend:       {backend}")
print(f"GPU present:   {has_gpu}")
print(f"CPU cores:     {cores}")
print(f"AVX2:          {avx2}")
print(f"AVX-512:       {avx512}")
print()

n_test = 6_000
will_ivf = info["recommend_ivf_pq"] == "true"
if has_gpu:
    print(f"write_batch_auto at N={n_test}: → IVF-PQ (GPU k-means, ~200k vec/s)")
elif cores >= 8 and n_test >= 5000:
    print(f"write_batch_auto at N={n_test}: → IVF-PQ (CPU k-means, {cores} cores)")
else:
    print(f"write_batch_auto at N={n_test}: → HNSW (best for CPU, small batch)")

if avx512:
    print("Distance kernel: AVX-512 + F16C (fastest CPU path)")
elif avx2:
    print("Distance kernel: AVX2 (good CPU path)")
else:
    print("Distance kernel: scalar fallback (no SIMD detected)")


## 7. Large-scale deferred write (GPU recommended)

`write_batch_auto_deferred` is designed for high-throughput ingest pipelines.
It writes Parquet immediately at full disk speed, then trains and embeds the index
in a background Tokio task — the commit returns fast.

Suitable for:
- Stream ingestion (Flink, Kafka consumers)
- Batch ETL pipelines that can't block on index training
- GPU k-means training for large codebooks (millions of vectors)


In [ ]:
DEFERRED_GPU_PATH = "/data/gpu_deferred_demo"
N_LARGE = 20_000

rng = np.random.default_rng(99)
embs_large  = rng.standard_normal((N_LARGE, DIM)).astype(np.float32)
texts_large = [f"deferred_{i}" for i in range(N_LARGE)]

os.makedirs(DEFERRED_GPU_PATH, exist_ok=True)
w = ailake.TableWriter(DEFERRED_GPU_PATH, dim=DIM, metric="cosine")

print(f"Starting deferred write for {N_LARGE} vectors...")
t0 = time.perf_counter()
w.write_batch_auto_deferred(texts_large, embs_large.tolist())
snap_id = w.commit()
t_commit = time.perf_counter() - t0

print(f"Parquet committed in {t_commit:.2f}s  ({N_LARGE/t_commit:,.0f} vec/s)")
print(f"snapshot_id = {snap_id}")
print()
print("Index training running in background. Status transitions:")
print("  IndexStatus::Pending → Indexing → Ready")
print()
print("Search on this table will block briefly until IndexStatus::Ready.")
print("Use 'ailake info <path>' to poll status from CLI.")


## Summary

| Feature | GPU (NVIDIA/AMD) | CPU SIMD (≥ 8 cores) | CPU (small) |
|---------|-----------------|----------------------|-------------|
| `write_batch_auto` | IVF-PQ / GPU k-means | IVF-PQ / CPU k-means | HNSW |
| Write throughput | ~200k vec/s (Parquet immediate, deferred) | ~200k vec/s (deferred) | ~50k vec/s |
| Distance kernel | cuBLAS / hipBLAS SGEMM | AVX-512 SIMD | Scalar / AVX2 |
| `hardware_info()` | backend=nvidia-cuda or amd-rocm | backend=cpu-simd | backend=cpu-simd |
| Fallback | Always available | — | — |

Run `--profile gpu` in Docker Compose to enable the NVIDIA runtime and test GPU paths end-to-end.
